## 設定とモデルのロード
設定値（モデル・層・成果物の場所）は `run.py` の先頭にまとめてある。変える場合はそちらを編集する。

In [ ]:
import run   # backend/ に移動し、sys.path を設定する
from run import (MODEL_ID, RESEARCH_MODEL_KEY, TARGET_LAYER, ASSETS_ROOT,
                 build_manager, start_voicevox, start_server)
from app.core.llm_engine import LLMEngine

print("model :", MODEL_ID)
print("layer :", TARGET_LAYER, "(hidden_states のインデックス)")
print("assets:", ASSETS_ROOT)

model : tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.5
layer : 21 (hidden_states のインデックス)
assets: /workspace/app_labGuide_v2/backend/data/processed/steering_assets


In [ ]:
# 量子化しない（活性化の値が変わり、成果物の avg_norm・α上限と対応しなくなる）
engine = LLMEngine(model_id=MODEL_ID, quantization_8bit=False)

Loading model: tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.5...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully.


## 成果物の検証

### 確認

In [ ]:
import numpy as np
d = np.load("data/processed/steering_assets/emotion_probe/probe_layer21.npz", allow_pickle=True)
print(d["dataset"], float(d["test_accuracy"]))    # synthetic_resp 0.8827... なら同梱品

synthetic_resp 0.8827948910593538


In [ ]:
# !touch scripts/__init__.py
# どこかに __init__.py 付きの scripts があるとそちらが先に見つかってしまってエラーになるのでinitを作成、中身は空

### A ~ Fの検証

In [ ]:
## 出力は/tmp
# A. 合成データ6,651文を生成条件で読み、ベクトルとプローブを作り直す
from scripts.build_assets import build_all
build_all(engine, steps=["activations", "vectors", "probe"], assets_root="/tmp/rebuilt")

[extract_activations] model=llama-3.1-swallow-8b-instruct-v0.5 position=response_mean layers=[21]
[load_synthetic] joy          : 739件 (level>=1)
[load_synthetic] sadness      : 739件 (level>=1)
[load_synthetic] anticipation : 739件 (level>=1)
[load_synthetic] surprise     : 739件 (level>=1)
[load_synthetic] anger        : 739件 (level>=1)
[load_synthetic] fear         : 739件 (level>=1)
[load_synthetic] disgust      : 739件 (level>=1)
[load_synthetic] trust        : 739件 (level>=1)

[extract_activations] 'anger' を抽出中... (739件)
[extract@response]   batch 10/47
[extract@response]   batch 20/47
[extract@response]   batch 30/47
[extract@response]   batch 40/47
[extract@response]   batch 47/47

[extract_activations] 'anticipation' を抽出中... (739件)
[extract@response]   batch 10/47
[extract@response]   batch 20/47
[extract@response]   batch 30/47
[extract@response]   batch 40/47
[extract@response]   batch 47/47

[extract_activations] 'disgust' を抽出中... (739件)
[extract@response]   batch 10/47
[extract

{'activations': {'dataset': 'synthetic_resp',
  'neutral_dir_name': 'neutral_resp',
  'layers': [21]},
 'vectors': ['/tmp/rebuilt/steering_vectors/llama-3.1-swallow-8b-instruct-v0.5/layer_21'],
 'probe': {'path': '/tmp/rebuilt/emotion_probe/probe_layer21.npz',
  'train_accuracy': 0.993421052631579,
  'test_accuracy': 0.8827948910593538,
  'classes': ['anger',
   'anticipation',
   'disgust',
   'fear',
   'joy',
   'neutral',
   'sadness',
   'surprise',
   'trust']}}

In [ ]:
# B. 同じ6,651文を理解条件で読み、プローブだけ作る
build_all(engine, steps=["activations", "probe"], position="prompt_last",
          assets_root="/tmp/rebuilt_pl")

[extract_activations] model=llama-3.1-swallow-8b-instruct-v0.5 position=prompt_last layers=[21]
[load_synthetic] joy          : 739件 (level>=1)
[load_synthetic] sadness      : 739件 (level>=1)
[load_synthetic] anticipation : 739件 (level>=1)
[load_synthetic] surprise     : 739件 (level>=1)
[load_synthetic] anger        : 739件 (level>=1)
[load_synthetic] fear         : 739件 (level>=1)
[load_synthetic] disgust      : 739件 (level>=1)
[load_synthetic] trust        : 739件 (level>=1)

[extract_activations] 'anger' を抽出中... (739件)
[extract_activations] 入力例:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

あなたは誠実で優秀な日本人のアシスタントです。<|eot_id|><|begin_of_text|><|start_header_id|>user<|end_header_id|>

あー、またこれ着なきゃいけないのか。<|eot_id|><|start_header_id|>

[extract_activations]   batch 10/47 処理済み
[extract_activations]   batch 20/47 処理済み
[extract_activations]   batch 30/47 処理済み
[extract_activations]   batch 40/47 処理済み
[extract_activations]   batch 47/47 処理済み

[extract_activations] 'anticipation' を抽

{'activations': {'dataset': 'synthetic_promptlast',
  'neutral_dir_name': 'neutral',
  'layers': [21]},
 'probe': {'path': '/tmp/rebuilt_pl/emotion_probe/probe_layer21.npz',
  'train_accuracy': 0.9924812030075187,
  'test_accuracy': 0.8707738542449286,
  'classes': ['anger',
   'anticipation',
   'disgust',
   'fear',
   'joy',
   'neutral',
   'sadness',
   'surprise',
   'trust']}}

In [ ]:
# C. Aで作ったベクトルを同梱品と比べる（再現確認）
from scripts.evaluate_probes import evaluate_probes, compare_assets
compare_assets("data/processed/steering_assets", "/tmp/rebuilt")
# 最後の行に 最小コサイン = 0.99以上 → 移植は正しい と出れば合格

[compare] 同梱: data/processed/steering_assets
[compare] 再生成: /tmp/rebuilt

感情                  コサイン       ノルム(同梱)        ノルム(再生成)
anger             1.0000         1.673           1.673
anticipation      1.0000         1.378           1.378
disgust           1.0000         1.281           1.281
fear              1.0000         1.243           1.243
joy               1.0000         1.370           1.370
sadness           1.0000         1.346           1.346
surprise          1.0000         2.075           2.076
trust             1.0000         1.739           1.740

項目                            同梱         再生成
avg_norm                 10.6073     10.6057
PCA 成分数                       14          14

[compare] 最小コサイン = 1.0000 → 移植は正しい


{'cosine': {'anger': 0.9999947883395752,
  'anticipation': 0.9999948566002319,
  'disgust': 0.9999949676234637,
  'fear': 0.9999917823035472,
  'joy': 0.999990261051583,
  'sadness': 0.9999943107676014,
  'surprise': 0.9999933871148723,
  'trust': 0.9999957132521001},
 'min_cosine': 0.999990261051583,
 'verdict': '移植は正しい'}

In [ ]:
# D. 3つのプローブで評価セット46文を判定し、比べる
evaluate_probes(engine, {
    "同梱（生成条件）": "data/processed/steering_assets/emotion_probe/probe_layer21.npz",
    "再生成（生成条件）": "/tmp/rebuilt/emotion_probe/probe_layer21.npz",
    "理解条件": "/tmp/rebuilt_pl/emotion_probe/probe_layer21.npz",
})

[eval] 評価セット 46件 読み込み: /workspace/app_labGuide_v2/backend/data/eval/probe_eval_set.jsonl
[eval] ラベルごとの件数: {'anger': 5, 'anticipation': 5, 'disgust': 5, 'fear': 5, 'joy': 5, 'neutral': 6, 'sadness': 5, 'surprise': 5, 'trust': 5}

[eval] 同梱（生成条件）: data/processed/steering_assets/emotion_probe/probe_layer21.npz
[EmotionProbe] data/processed/steering_assets/emotion_probe/probe_layer21.npz を読み込みました (layer=21, dataset=synthetic_resp, position=response_mean, test_acc=0.883, classes=9)
[eval] ラベルごとの正解数: anger 5/5, anticipation 4/5, disgust 1/5, fear 3/5, joy 3/5, neutral 3/6, sadness 4/5, surprise 2/5, trust 5/5

[eval] 再生成（生成条件）: /tmp/rebuilt/emotion_probe/probe_layer21.npz
[EmotionProbe] /tmp/rebuilt/emotion_probe/probe_layer21.npz を読み込みました (layer=21, dataset=synthetic_resp, position=response_mean, test_acc=0.883, classes=9)
[eval] ラベルごとの正解数: anger 5/5, anticipation 4/5, disgust 1/5, fear 3/5, joy 4/5, neutral 3/6, sadness 4/5, surprise 2/5, trust 5/5

[eval] 理解条件: /tmp/rebuilt_pl/emotion_pro

{'summary': {'同梱（生成条件）': {'acc_all': 0.6521739130434783,
   'acc_emotion': 0.675,
   'acc_neutral': 0.5,
   'neutral_misfire': 0.5,
   'mean_true_prob': 0.6304334550555011,
   'probe_test_acc': 0.8827948910593538},
  '再生成（生成条件）': {'acc_all': 0.6739130434782609,
   'acc_emotion': 0.7,
   'acc_neutral': 0.5,
   'neutral_misfire': 0.5,
   'mean_true_prob': 0.6289858332351208,
   'probe_test_acc': 0.8827948910593538},
  '理解条件': {'acc_all': 0.7391304347826086,
   'acc_emotion': 0.825,
   'acc_neutral': 0.16666666666666666,
   'neutral_misfire': 0.8333333333333334,
   'mean_true_prob': 0.6872124708796545,
   'probe_test_acc': 0.8707738542449286}},
 'results': {'同梱（生成条件）': [{'pred': 'neutral',
    'prob': 0.44290342926979065,
    'true_prob': 0.4426365792751312,
    'correct': False},
   {'pred': 'anger',
    'prob': 0.5940212607383728,
    'true_prob': 0.13094878196716309,
    'correct': False},
   {'pred': 'joy',
    'prob': 0.9956291913986206,
    'true_prob': 0.9956291913986206,
    'corr

In [ ]:
# E. 比較結果の集計（D が保存した CSV から計算する。数字の出どころを残すためのセル）
import glob
import pandas as pd

csv_path = sorted(glob.glob("data/eval/results/probe_eval_*.csv"))[-1]   # 最新の結果
df = pd.read_csv(csv_path, encoding="utf-8-sig")
names = [c[:-len(":pred")] for c in df.columns if c.endswith(":pred")]   # プローブの表示名
is_neu = df["label"] == "neutral"
print("集計に使ったCSV:", csv_path)

# (1) 感情文・中立文ごとの正解数と正解率
rows = []
for nm in names:
    ok = df[f"{nm}:correct"] == 1
    rows.append({"プローブ": nm,
                 "感情文": f"{ok[~is_neu].sum()}/{(~is_neu).sum()}", "感情文 正解率": round(ok[~is_neu].mean(), 3),
                 "中立文": f"{ok[is_neu].sum()}/{is_neu.sum()}",   "中立文 正解率": round(ok[is_neu].mean(), 3),
                 "全体 正解率": round(ok.mean(), 3)})
display(pd.DataFrame(rows))

# (2) 中立文を各プローブがどう判定したか（理解条件が質問を anticipation と読む傾向の確認）
display(df.loc[is_neu, ["text"] + [f"{nm}:pred" for nm in names]])

# (3) 最初と最後のプローブで、正誤が分かれた文
a, b = names[0], names[-1]
diff = df[df[f"{a}:correct"] != df[f"{b}:correct"]]
print(f"正誤が分かれた文: {len(diff)}件（{a} と {b}）")
display(diff[["text", "label", f"{a}:pred", f"{b}:pred"]])

集計に使ったCSV: data/eval/results/probe_eval_20260928_071555.csv


,プローブ,感情文,感情文 正解率,中立文,中立文 正解率,全体 正解率
0,同梱（生成条件）,27/40,0.675,3/6,0.500,0.652
1,再生成（生成条件）,28/40,0.700,3/6,0.500,0.674
2,理解条件,33/40,0.825,1/6,0.167,0.739


,text,同梱（生成条件）:pred,再生成（生成条件）:pred,理解条件:pred
40,大学の学食について教えてください,neutral,neutral,anticipation
41,こんにちは,joy,joy,joy
42,こんいちは,trust,trust,joy
43,明日の天気はどうなるかな,anticipation,anticipation,anticipation
44,研究室の場所を教えてもらえますか,neutral,neutral,anticipation
45,今日の昼ごはん何食べた？,neutral,neutral,neutral


正誤が分かれた文: 12件（同梱（生成条件） と 理解条件）


,text,label,同梱（生成条件）:pred,理解条件:pred
0,第一志望の会社から内定もらえた！,joy,neutral,joy
6,頑張ったのに試験に落ちてしまった,sadness,anger,sadness
17,まさかあのチームが優勝するとは思わなかった,surprise,joy,surprise
19,知らない番号から電話が来たと思ったら、高校の同級生だった,surprise,neutral,surprise
25,夜道を歩いてたら、後ろからずっと足音がついてくるんだ,fear,neutral,fear
26,明日の手術のことを考えると不安で眠れない,fear,fear,anticipation
29,さっきから部屋で変な物音がするんだけど,fear,anger,fear
30,冷蔵庫の奥から腐った弁当が出てきて最悪,disgust,neutral,disgust
31,人の悪口ばかり言う人とは関わりたくない,disgust,disgust,anger
33,排水溝にぬめりがびっしりで気持ち悪い,disgust,anger,disgust


In [ ]:
# F. 大まかな方向の評価（感情の正負・プルチックの環での近さ）
#    アプリは確率ベクトル全体を内部状態に足すので、1位の正誤だけでなく「どの方向に傾いたか」も評価する
import numpy as np
import pandas as pd
from app.core.emotion_probe import EmotionProbe
from scripts.evaluate_probes import load_eval_set

# プルチックの環の順（45°ずつ並ぶ。EmotionDynamics と同じ順序）
WHEEL = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
# 感情の正負（前提: anticipation は正、surprise は正負どちらでもない 0 とした）
VALENCE = {"joy": +1, "trust": +1, "anticipation": +1, "surprise": 0,
           "fear": -1, "sadness": -1, "disgust": -1, "anger": -1}
# 環上の単位ベクトル（joy を 0°として45°ずつ）
U = {e: np.array([np.cos(2 * np.pi * i / 8), np.sin(2 * np.pi * i / 8)]) for i, e in enumerate(WHEEL)}

probes = {
    "生成条件（同梱）": "data/processed/steering_assets/emotion_probe/probe_layer21.npz",
    "理解条件": "/tmp/rebuilt_pl/emotion_probe/probe_layer21.npz",
}
items = load_eval_set()
records = []
for name, path in probes.items():
    probe = EmotionProbe(WHEEL, engine, probe_path=path)
    for it in items:
        p = dict(zip(WHEEL, probe.predict_emotion(it["text"])))   # neutral を除く8感情の確率
        vec = sum(p[e] * U[e] for e in WHEEL)                     # 予測を環上の1点にまとめる
        val = sum(p[e] * VALENCE[e] for e in WHEEL)               # 正負の傾き（+なら正、-なら負）
        rec = {"probe": name, "text": it["text"], "label": it["label"],
               "neutral_prob": probe.neutral_score(it["text"]), "valence": val}
        if it["label"] != "neutral":
            # 環コサイン: 予測の方向と正解の方向の cos（1=一致, 0.71=隣, 0=90°ずれ, -1=反対）
            rec["wheel_cos"] = float(vec @ U[it["label"]] / (np.linalg.norm(vec) + 1e-12))
            v = VALENCE[it["label"]]
            # 正負一致: 正解の正負がある文（surprise 以外）だけ判定する
            rec["valence_ok"] = float(np.sign(val) == v) if v != 0 else np.nan
        records.append(rec)
df_dir = pd.DataFrame(records)

# 集計（感情文40件。中立文は neutral 確率の平均だけ見る）
emo = df_dir[df_dir["label"] != "neutral"]
summary = emo.groupby("probe").agg(
    環コサイン平均=("wheel_cos", "mean"),
    隣接まで許容した正解率=("wheel_cos", lambda s: (s >= 0.7).mean()),   # cos45° ≒ 0.71
    正負一致率=("valence_ok", "mean"))                                   # surprise の文は除外される
summary["中立文のneutral確率平均"] = df_dir[df_dir["label"] == "neutral"].groupby("probe")["neutral_prob"].mean()
display(summary.round(3))

# 1文ごとの環コサイン（どの文が大きく外れたか）
display(emo.pivot_table(index=["label", "text"], columns="probe", values="wheel_cos").round(2))
df_dir.to_csv("data/eval/results/probe_eval_direction.csv", index=False, encoding="utf-8-sig")

[eval] 評価セット 46件 読み込み: /workspace/app_labGuide_v2/backend/data/eval/probe_eval_set.jsonl
[eval] ラベルごとの件数: {'anger': 5, 'anticipation': 5, 'disgust': 5, 'fear': 5, 'joy': 5, 'neutral': 6, 'sadness': 5, 'surprise': 5, 'trust': 5}
[EmotionProbe] data/processed/steering_assets/emotion_probe/probe_layer21.npz を読み込みました (layer=21, dataset=synthetic_resp, position=response_mean, test_acc=0.883, classes=9)
[EmotionProbe] /tmp/rebuilt_pl/emotion_probe/probe_layer21.npz を読み込みました (layer=21, dataset=synthetic_promptlast, position=prompt_last, test_acc=0.871, classes=9)


,環コサイン平均,隣接まで許容した正解率,正負一致率,中立文のneutral確率平均
probe,,,,
理解条件,0.897,0.950,0.943,0.153
生成条件（同梱）,0.802,0.825,0.971,0.422


probe                                      理解条件  生成条件（同梱）
label        text                                        
anger        この対応は本当にひどい                   1.00      1.00
             何度言っても約束を守らないのはどういうこと？        1.00      1.00
             勝手に人のデータを消すなんて信じられない          1.00      1.00
             説明もなしに予定を変えられて腹が立つ            1.00      1.00
             順番抜かしされて、正直かなり頭にきた            1.00      1.00
anticipation 夏休みに何しようか今から考えてるんだ            1.00      1.00
             明日の発表、どんな反応がもらえるか楽しみ          1.00      1.00
             来週ついに新しいゲームが発売されるんだ           0.94      1.00
             次の実験結果が出るのが待ち遠しい              1.00      1.00
             週末に友達と遊園地に行く予定なんだ             0.99      0.86
disgust      人の悪口ばかり言う人とは関わりたくない           0.86      0.97
             冷蔵庫の奥から腐った弁当が出てきて最悪           1.00      0.65
             嘘をついて人を陥れるなんて、見ていて気分が悪くなる    -0.39      0.57
             排水溝にぬめりがびっしりで気持ち悪い            1.00      0.93
             電車で隣の人がずっと大声で電話してて不快だった       0.71      0.72
fear         さっきから部屋で変な物音がするんだけど           1.00     -1.00
             夜道を歩いてたら、後ろからずっと足音がついてくるんだ    1.00      1.00
             大きい地震が来たらどうしようって最近よく考える       1.00      0.99
             明日の手術のことを考えると不安で眠れない         -0.79      0.95
             締め切りに間に合わなかったらどうなるか怖い         1.00      1.00
joy          この前話してた本、やっと読み終わったよ           0.99      0.20
             ずっと悩んでた実験がやっとうまくいったんだ         1.00      1.00
             久しぶりに家族でご飯に行けて楽しかった           1.00      1.00
             今日は天気が良くて気分がいいな               1.00      1.00
             第一志望の会社から内定もらえた！              1.00      1.00
sadness      仲の良かった友達が遠くに引っ越しちゃう           1.00      1.00
             最近なんだか何をしても楽しくないんだ            1.00      0.75
             楽しみにしてた旅行が中止になっちゃった           0.99      1.00
             頑張ったのに試験に落ちてしまった              0.80      0.62
             飼ってた猫が昨日死んじゃったんだ              1.00      1.00
surprise     えっ、先生が急に辞めるって本当？              1.00      1.00
             まさかあのチームが優勝するとは思わなかった         1.00     -0.15
             知らない番号から電話が来たと思ったら、高校の同級生だった  1.00      0.87
             起きたら外が一面の雪でびっくりした             0.81      1.00
             道を歩いてたら偶然、昔の友達にばったり会った        0.99     -0.82
trust        あなたに相談すると、いつも安心できるよ           1.00      1.00
             このお店の店長さんは本当に信頼できる人なんだ        1.00      1.00
             先輩に任せておけば大丈夫だと思ってる            1.00      1.00
             困ったときは家族がいるから心強い              1.00      1.00
             彼なら約束は必ず守ってくれる                1.00      1.00